<a href="https://colab.research.google.com/github/inoue0426/llm-tuning-playground/blob/main/notebooks/17_ctd_ipo_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 17 — CTD IPO Follow-up

IPO-only follow-up to the boundary-fixed DPO experiment. The goal is to test whether **Identity Preference Optimization (IPO)** reduces the seed-dependent answer/abstain polarization observed with DPO when preference labels are deterministic.

Design: same Qwen2.5-0.5B base model, same saved Robust SFT adapters from Experiment 15, same three entity-disjoint splits, same three seeds, same preference-pair construction, same 60 optimization steps, and the same token-boundary preflight as Experiment 16. The main intended change is the preference loss: `loss_type='ipo'`.

Research question: **Can IPO preserve evidence selection while adding evidence-sufficiency detection more stably than DPO?**


In [1]:
!pip -q install -U transformers datasets trl peft accelerate bitsandbytes sentencepiece requests

import os, re, gc, json, random, gzip, inspect
from pathlib import Path
import numpy as np, pandas as pd, torch, requests
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import PeftModel, prepare_model_for_kbit_training
from trl import DPOConfig, DPOTrainer

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
SEEDS = [1,2,3]
SPLITS = ['ChemicalID','GeneID','DiseaseID']
N_TRAIN = 1500
N_EVAL = 100
IPO_STEPS = 60
IPO_BETA = 0.1
IPO_LR = 5e-6

ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
DATA_DIR = ROOT/'ctd_data'; DATA_DIR.mkdir(parents=True, exist_ok=True)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/llm-tuning-playground')
except Exception:
    DRIVE_ROOT = ROOT/'llm-tuning-playground'

ROBUST_DIR = DRIVE_ROOT/'results/15/adapters'
RESULT_DIR = DRIVE_ROOT/'results/17'
ADAPTER_DIR = RESULT_DIR/'adapters'
RESULT_DIR.mkdir(parents=True, exist_ok=True); ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
RESULT_CSV = RESULT_DIR/'17_ipo_results.csv'
SUMMARY_CSV = RESULT_DIR/'17_ipo_summary.csv'
CONFIG_JSON = RESULT_DIR/'17_ipo_config.json'

config = dict(model=MODEL_NAME,seeds=SEEDS,splits=SPLITS,n_train=N_TRAIN,n_eval=N_EVAL,ipo_steps=IPO_STEPS,beta=IPO_BETA,learning_rate=IPO_LR,loss_type='ipo',source_robust_adapters=str(ROBUST_DIR),pair_construction='same as boundary-fixed DPO',boundary_fix='leading-space completion + exact token-prefix preflight')
CONFIG_JSON.write_text(json.dumps(config,indent=2),encoding='utf-8')
print('CUDA:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Results:', RESULT_DIR)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 146.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 52.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Mounted at /content/drive
CUDA: NVIDIA L4
Results: /content/drive/MyDrive/llm-tuning-playground/results/17


In [2]:
# CTD acquisition and parsing
CHEM_NAME='CTD_chem_gene_ixns.tsv.gz'
GD_NAMES=['CTD_curated_genes_diseases.tsv.gz','CTD_genes_diseases.tsv.gz']

def valid_gzip(path,min_bytes=10000):
    path=Path(path)
    if not path.exists() or path.stat().st_size<min_bytes: return False
    try:
        with open(path,'rb') as f:
            if f.read(2)!=b'\x1f\x8b': return False
        with gzip.open(path,'rb') as f: f.read(256)
        return True
    except Exception: return False

def find_local(name):
    for p in [Path.cwd()/name,ROOT/name,DATA_DIR/name,Path('/content/drive/MyDrive')/name,Path('/content/drive/MyDrive/ctd')/name,Path('/content/drive/MyDrive/data')/name]:
        if valid_gzip(p): print('Found:',p); return p
    return None

def download_ctd(name):
    dest=DATA_DIR/name
    for url in [f'https://ctdbase.org/reports/{name}',f'https://ctdbase.org/downloads/{name}',f'http://ctdbase.org/reports/{name}']:
        try:
            print('Trying:',url)
            with requests.get(url,stream=True,timeout=(20,300),allow_redirects=True,headers={'User-Agent':'Mozilla/5.0'}) as r:
                r.raise_for_status()
                with open(dest,'wb') as f:
                    for ch in r.iter_content(1024*1024):
                        if ch: f.write(ch)
            if valid_gzip(dest): print('Downloaded:',dest); return dest
        except Exception as e: print(' failed:',type(e).__name__,str(e)[:120])
        dest.unlink(missing_ok=True)
    return None

def ensure_ctd(names):
    if isinstance(names,str): names=[names]
    for n in names:
        p=find_local(n)
        if p: return p
    for n in names:
        p=download_ctd(n)
        if p: return p
    raise FileNotFoundError('Could not obtain CTD data: '+', '.join(names))

def read_ctd(path,expected_any):
    header=None
    with gzip.open(path,'rt',encoding='utf-8',errors='replace') as f:
        for line in f:
            if not line.startswith('#'): break
            s=line.lstrip('#').strip()
            if '\t' in s:
                cols=[x.strip() for x in s.split('\t')]
                if any(x in cols for x in expected_any): header=cols
    if header is None: raise ValueError(f'Could not recover CTD header from {path}')
    return pd.read_csv(path,sep='\t',comment='#',compression='gzip',dtype=str,low_memory=False,header=None,names=header)

def pick(df,names):
    for n in names:
        if n in df.columns: return n
    raise KeyError(f'None of {names} found')

CHEM_GENE=ensure_ctd(CHEM_NAME); GENE_DISEASE=ensure_ctd(GD_NAMES)
cg=read_ctd(CHEM_GENE,['ChemicalName','ChemicalID','GeneSymbol','GeneID'])
gd=read_ctd(GENE_DISEASE,['GeneSymbol','GeneID','DiseaseName','DiseaseID'])
c_name=pick(cg,['ChemicalName']); c_id=pick(cg,['ChemicalID']); g_sym1=pick(cg,['GeneSymbol']); g_id1=pick(cg,['GeneID'])
g_sym2=pick(gd,['GeneSymbol']); g_id2=pick(gd,['GeneID']); d_name=pick(gd,['DiseaseName']); d_id=pick(gd,['DiseaseID'])
cg2=cg[[c_name,c_id,g_sym1,g_id1]].dropna().drop_duplicates(); gd2=gd[[g_sym2,g_id2,d_name,d_id]].dropna().drop_duplicates()
cg2.columns=['ChemicalName','ChemicalID','GeneSymbol','GeneID']; gd2.columns=['GeneSymbol','GeneID','DiseaseName','DiseaseID']
paths=cg2.merge(gd2,on=['GeneSymbol','GeneID'],how='inner').drop_duplicates()
paths=paths[(paths.ChemicalName.str.len()<100)&(paths.DiseaseName.str.len()<120)].reset_index(drop=True)
edge_pool=gd2[['GeneSymbol','DiseaseName']].drop_duplicates().reset_index(drop=True)
assert len(paths)>3000
print('Two-hop paths:',len(paths))


Trying: https://ctdbase.org/reports/CTD_chem_gene_ixns.tsv.gz
Downloaded: /content/ctd_data/CTD_chem_gene_ixns.tsv.gz
Trying: https://ctdbase.org/reports/CTD_curated_genes_diseases.tsv.gz
Downloaded: /content/ctd_data/CTD_curated_genes_diseases.tsv.gz
Two-hop paths: 9707313


In [3]:
# Controlled benchmark helpers
def render_prompt(row,edges):
    lines=[f'- {g} -> {d}' for g,d in edges]
    return ('Use only the supplied evidence. Determine the disease supported by the path from the queried chemical through the queried gene. '
            'If no supplied gene-disease relation supports the queried gene, answer exactly: No supported path.\n\n'
            f'Chemical: {row.ChemicalName}\nGene: {row.GeneSymbol}\nEvidence:\n'+'\n'.join(lines))

def positive_edges(row,k,rng):
    edges=[(str(row.GeneSymbol),str(row.DiseaseName))]
    pool=edge_pool[(edge_pool.GeneSymbol!=row.GeneSymbol)&(edge_pool.DiseaseName!=row.DiseaseName)]
    if k:
        sub=pool.sample(n=k,random_state=rng.randint(0,2**31-1)); edges += [(str(g),str(d)) for g,d in sub.itertuples(index=False,name=None)]
    rng.shuffle(edges); return edges

def no_path_edges(row,k,rng,lexical=False):
    pool=edge_pool[(edge_pool.GeneSymbol!=row.GeneSymbol)&(edge_pool.DiseaseName!=row.DiseaseName)].copy(); edges=[]
    if lexical:
        sym=str(row.GeneSymbol); near=pool[pool.GeneSymbol.astype(str).str.startswith(sym[:max(1,min(2,len(sym)))])]
        if len(near):
            x=near.sample(1,random_state=rng.randint(0,2**31-1)).iloc[0]; edges.append((str(x.GeneSymbol),str(x.DiseaseName))); pool=pool[pool.GeneSymbol!=x.GeneSymbol]
    need=k-len(edges)
    if need>0:
        sub=pool.sample(need,random_state=rng.randint(0,2**31-1)); edges += [(str(g),str(d)) for g,d in sub.itertuples(index=False,name=None)]
    rng.shuffle(edges); return edges

def counterfactual_edges(row,rng):
    c=edge_pool[(edge_pool.GeneSymbol==row.GeneSymbol)&(edge_pool.DiseaseName!=row.DiseaseName)]
    cf=str(c.sample(1,random_state=rng.randint(0,2**31-1)).iloc[0].DiseaseName) if len(c) else str(edge_pool[edge_pool.DiseaseName!=row.DiseaseName].sample(1,random_state=rng.randint(0,2**31-1)).iloc[0].DiseaseName)
    return [(str(row.GeneSymbol),cf)],cf

def answer_text(row): return f'Disease: {row.DiseaseName}. Reasoning: {row.ChemicalName} -> {row.GeneSymbol} -> {row.DiseaseName}.'

def make_split(df,col,seed):
    r=np.random.default_rng(seed); ents=df[col].dropna().unique().copy(); r.shuffle(ents); cut=max(1,int(.8*len(ents)))
    tr_e,te_e=set(ents[:cut]),set(ents[cut:]); trp=df[df[col].isin(tr_e)]; tep=df[df[col].isin(te_e)].drop_duplicates(['ChemicalID','GeneID','DiseaseID'])
    assert len(trp)>=N_TRAIN and len(tep)>=N_EVAL
    tr=trp.sample(N_TRAIN,random_state=seed).reset_index(drop=True); te=tep.sample(N_EVAL,random_state=1000+seed).reset_index(drop=True)
    assert set(tr[col]).isdisjoint(set(te[col])); return tr,te

def item(row,edges,target,typ): return {'target_gene':str(row.GeneSymbol),'target_disease':None if target is None else str(target),'evidence_edges':[(str(g),str(d)) for g,d in edges],'prompt':render_prompt(row,edges),'answer_type':typ}

def make_eval_sets(df,seed):
    rng=random.Random(20000+seed); out={k:[] for k in ['clean','distractor_5','hard_no_path','lexical_no_path','counterfactual']}
    for _,row in df.iterrows():
        out['clean'].append(item(row,positive_edges(row,0,rng),row.DiseaseName,'positive'))
        out['distractor_5'].append(item(row,positive_edges(row,5,rng),row.DiseaseName,'positive'))
        out['hard_no_path'].append(item(row,no_path_edges(row,5,rng,False),None,'no_path'))
        out['lexical_no_path'].append(item(row,no_path_edges(row,5,rng,True),None,'no_path'))
        e,cf=counterfactual_edges(row,rng); out['counterfactual'].append(item(row,e,cf,'positive'))
    return out

def norm(s): return re.sub(r'\s+',' ',str(s).strip().lower())
def score_one(x,p):
    p=norm(p)
    if x['answer_type']=='no_path': return 'no supported path' in p
    return norm(x['target_disease']) in p and 'no supported path' not in p
def score_set(items,preds): return float(np.mean([score_one(x,p) for x,p in zip(items,preds)]))


In [4]:
# Preference pairs: intentionally identical to Experiment 16, including the leading-space boundary fix
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME,use_fast=True)
tokenizer.pad_token=tokenizer.pad_token or tokenizer.eos_token
tokenizer.padding_side='left'

def make_pref_dataset(df,seed):
    rng=random.Random(30000+seed); rec=[]
    for _,row in df.iterrows():
        if rng.random() < 0.5:
            edges=no_path_edges(row,rng.choice([3,5,10]),rng,rng.random()<0.5)
            rejected_disease=edges[0][1]
            chosen='No supported path.'
            rejected=f'Disease: {rejected_disease}.'
        else:
            edges=positive_edges(row,rng.choice([1,3,5,10]),rng)
            distractors=[d for g,d in edges if g != str(row.GeneSymbol)]
            rejected_disease=distractors[0] if distractors else 'Unknown disease'
            chosen=answer_text(row)
            rejected=f'Disease: {rejected_disease}.'
        prompt=render_prompt(row,edges)+'\nAnswer:'
        rec.append({'prompt':prompt,'chosen':' '+chosen,'rejected':' '+rejected})
    return Dataset.from_list(rec)

def boundary_preflight(ds,n=100):
    n=min(n,len(ds)); bad=[]
    for i in range(n):
        ex=ds[i]; p=tokenizer(ex['prompt'],add_special_tokens=False).input_ids
        for key in ('chosen','rejected'):
            full=tokenizer(ex['prompt']+ex[key],add_special_tokens=False).input_ids
            if full[:len(p)] != p: bad.append((i,key))
    if bad: raise RuntimeError(f'IPO token-boundary preflight FAILED: {bad[:10]}')
    print(f'IPO token-boundary preflight: {n}/{n} examples aligned for chosen + rejected ✅')

_tr,_=make_split(paths,'ChemicalID',SEEDS[0]); _preview=make_pref_dataset(_tr,SEEDS[0]); boundary_preflight(_preview,100)
print('Preference example:',_preview[0])


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

IPO token-boundary preflight: 100/100 examples aligned for chosen + rejected ✅
Preference example: {'prompt': 'Use only the supplied evidence. Determine the disease supported by the path from the queried chemical through the queried gene. If no supplied gene-disease relation supports the queried gene, answer exactly: No supported path.\n\nChemical: Particulate Matter\nGene: REL\nEvidence:\n- NAT1 -> Dermatitis, Occupational\n- REL -> Hypokinesia\n- ARL13B -> Joubert Syndrome 8\n- RET -> Multiple Endocrine Neoplasia\n- MIR6132 -> Heart Failure\n- MTCH2 -> Obesity\nAnswer:', 'chosen': ' Disease: Hypokinesia. Reasoning: Particulate Matter -> REL -> Hypokinesia.', 'rejected': ' Disease: Dermatitis, Occupational.'}


In [5]:
# Model/trainer/evaluation utilities
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,bnb_4bit_use_double_quant=True)

def load_policy_and_ref(adapter_path):
    base=AutoModelForCausalLM.from_pretrained(MODEL_NAME,quantization_config=bnb,device_map='auto',torch_dtype='auto')
    base.config.use_cache=False
    base=prepare_model_for_kbit_training(base)
    policy=PeftModel.from_pretrained(base,str(adapter_path),is_trainable=True)

    ref_base=AutoModelForCausalLM.from_pretrained(MODEL_NAME,quantization_config=bnb,device_map='auto',torch_dtype='auto')
    ref_base.config.use_cache=False
    ref=PeftModel.from_pretrained(ref_base,str(adapter_path),is_trainable=False)
    ref.eval()
    return policy,ref

def make_ipo_args(out_dir):
    kwargs=dict(output_dir=str(out_dir),max_steps=IPO_STEPS,per_device_train_batch_size=1,gradient_accumulation_steps=8,learning_rate=IPO_LR,beta=IPO_BETA,loss_type='ipo',logging_steps=20,save_strategy='no',report_to='none',remove_unused_columns=False,seed=0)
    sig=inspect.signature(DPOConfig.__init__).parameters
    if 'max_length' in sig: kwargs['max_length']=512
    if 'max_prompt_length' in sig: kwargs['max_prompt_length']=448
    return DPOConfig(**{k:v for k,v in kwargs.items() if k in sig})

@torch.inference_mode()
def generate_batch(model,items,batch_size=8,max_new_tokens=48):
    model.eval(); outs=[]
    for i in range(0,len(items),batch_size):
        ps=[x['prompt']+'\nAnswer:' for x in items[i:i+batch_size]]
        enc=tokenizer(ps,return_tensors='pt',padding=True,truncation=True,max_length=448).to(model.device)
        gen=model.generate(**enc,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
        for j,g in enumerate(gen):
            n=int(enc['attention_mask'][j].sum().item()); outs.append(tokenizer.decode(g[enc['input_ids'].shape[1]:],skip_special_tokens=True))
    return outs

def load_rows():
    if RESULT_CSV.exists():
        try: return pd.read_csv(RESULT_CSV).to_dict('records')
        except Exception: pass
    return []

def checkpoint(rows):
    tmp=RESULT_CSV.with_suffix('.tmp.csv'); pd.DataFrame(rows).to_csv(tmp,index=False); os.replace(tmp,RESULT_CSV)

def completed(rows,split,seed):
    got={r['condition'] for r in rows if r['split']==split and int(r['seed'])==int(seed) and r['method']=='ipo'}
    return len(got)==5


In [ ]:
# Run all 3 splits x 3 seeds. Completed runs resume from Drive.
rows=load_rows()
for split in SPLITS:
    for seed in SEEDS:
        print('\n'+'='*80); print('SPLIT',split,'SEED',seed); print('='*80)
        if completed(rows,split,seed):
            print('Already complete — skipping.'); continue
        set_seed(seed); random.seed(seed); np.random.seed(seed)
        tr,te=make_split(paths,split,seed); eval_sets=make_eval_sets(te,seed); pref_ds=make_pref_dataset(tr,seed); boundary_preflight(pref_ds,100)
        adapter_path=ROBUST_DIR/f'robust_{split}_{seed}'
        if not adapter_path.exists(): raise FileNotFoundError(f'Missing robust adapter: {adapter_path}')
        print('Training IPO from:',adapter_path)
        policy,ref=load_policy_and_ref(adapter_path)
        args=make_ipo_args(RESULT_DIR/'trainer_tmp')
        trainer=DPOTrainer(model=policy,ref_model=ref,args=args,train_dataset=pref_ds,processing_class=tokenizer)
        trainer.train()
        out_adapter=ADAPTER_DIR/f'ipo_{split}_{seed}'; policy.save_pretrained(out_adapter)
        existing={(r['split'],int(r['seed']),r['condition']) for r in rows if r['method']=='ipo'}
        for cond,items in eval_sets.items():
            key=(split,seed,cond)
            if key in existing: continue
            preds=generate_batch(policy,items); acc=score_set(items,preds)
            print(cond,acc)
            rows.append({'split':split,'seed':seed,'method':'ipo','condition':cond,'accuracy':acc,'beta':IPO_BETA,'steps':IPO_STEPS,'lr':IPO_LR})
            checkpoint(rows); print('Checkpointed rows:',len(rows))
        del trainer,policy,ref; gc.collect(); torch.cuda.empty_cache()

print('Done. Raw results:',RESULT_CSV)



SPLIT ChemicalID SEED 1
IPO token-boundary preflight: 100/100 examples aligned for chosen + rejected ✅
Training IPO from: /content/drive/MyDrive/llm-tuning-playground/results/15/adapters/robust_ChemicalID_1


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

Tokenizing train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
20,12.844385
40,5.146188
60,3.456139


clean 0.94
Checkpointed rows: 21
distractor_5 0.95
Checkpointed rows: 22
hard_no_path 0.0
Checkpointed rows: 23
lexical_no_path 0.0
Checkpointed rows: 24
counterfactual 0.97
Checkpointed rows: 25

SPLIT GeneID SEED 3


In [ ]:
# Aggregate summary + simple selection/sufficiency diagnostics
df=pd.read_csv(RESULT_CSV)
summary=(df.groupby(['split','method','condition']).accuracy.agg(['mean','std','count']).reset_index())
summary.to_csv(SUMMARY_CSV,index=False)
display(summary)

wide=df.pivot_table(index=['split','seed','method'],columns='condition',values='accuracy').reset_index()
wide['selection']=wide['distractor_5']
wide['sufficiency']=(wide['hard_no_path']+wide['lexical_no_path'])/2
wide['harmonic']=2*wide['selection']*wide['sufficiency']/(wide['selection']+wide['sufficiency']+1e-12)
display(wide.sort_values(['split','seed']))
print('Overall means:')
display(df.groupby('condition').accuracy.agg(['mean','std','count']))
print('Saved:',SUMMARY_CSV)
